 # BGE-M3

## 1. ติดตั้ง Dependencies

In [1]:
!pip install transformers>=4.51.0 sentence-transformers>=2.7.0 torch

## 2. Import Libraries

In [ ]:
import os
from sentence_transformers import SentenceTransformer
import torch
import re
import json
from pathlib import Path

os.environ["HF_TOKEN"] = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"

c:\Users\n.o_o.n\miniconda3\envs\ai6\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. เลือก GPU

In [3]:
GPU_ID = 1  # เลือก GPU1 (index 1)

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"จำนวน GPU ที่มี: {gpu_count}")
    for i in range(gpu_count):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

    if gpu_count > GPU_ID:
        device = torch.device(f"cuda:{GPU_ID}")
        print(f"กำลังใช้ GPU{GPU_ID}")
    else:
        device = torch.device("cuda:0")
        print(f"ไม่พบ GPU{GPU_ID} — ใช้ GPU0 แทน")
else:
    device = torch.device("cpu")
    print("ไม่มี GPU — ใช้ CPU")

print(f"\nกำลังโหลด BAAI/bge-m3...")
model = SentenceTransformer("BAAI/bge-m3", device=str(device))
print(f"โหลดสำเร็จ! Device: {model.device}")

จำนวน GPU ที่มี: 1
  GPU 0: NVIDIA GeForce RTX 3050 Laptop GPU
ไม่พบ GPU1 — ใช้ GPU0 แทน

กำลังโหลด BAAI/bge-m3...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 27933.45it/s]


โหลดสำเร็จ! Device: cuda:0


In [4]:
data_path = Path("data.txt")
raw_text = data_path.read_text(encoding="utf-8")
print(f"อ่านข้อมูลสำเร็จ: {len(raw_text)} ตัวอักษร")
print("\n--- ตัวอย่างข้อมูล (100 ตัวอักษรแรก) ---")
print(raw_text[:200])

อ่านข้อมูลสำเร็จ: 13514 ตัวอักษร

--- ตัวอย่างข้อมูล (100 ตัวอักษรแรก) ---
## สถานที่ทางศาสนา และวัด
* วัดปทุมวนารามราชวรวิหาร (BTS สยาม):
  * ตั้งอยู่ระหว่างสยามพารากอนและเซ็นทรัลเวิลด์ ก่อตั้งปี พ.ศ. 2400 ในรัชกาลที่ 4
  * ห่างจากสถานี BTS สยาม ทางออก 5 เพียง 520 เมตร (เดิ


In [5]:
def parse_tourism_data(text: str) -> list[dict]:
    """
    แยกข้อมูลสถานที่ท่องเที่ยวจาก raw text
    แต่ละ entry ประกอบด้วย: ชื่อสถานที่, หมวดหมู่, สถานี, รายละเอียด
    """
    places = []
    current_category = ""

    # จับหมวดหมู่หลัก
    category_pattern = re.compile(r"^##\s+(.+)$", re.MULTILINE)
    # จับรายการ bullet points (สถานที่แต่ละแห่ง)
    bullet_pattern = re.compile(r"^\*\s+(.+?)(?=^\*|^##|^---|\Z)", re.MULTILINE | re.DOTALL)

    # แบ่งตาม section
    sections = re.split(r"^---$", text, flags=re.MULTILINE)

    for section in sections:
        # หาหมวดหมู่
        cat_match = category_pattern.search(section)
        if cat_match:
            current_category = cat_match.group(1).strip()

        # หา bullet points
        for bullet in bullet_pattern.finditer(section):
            content = bullet.group(1).strip()
            # ดึงชื่อสถานที่และสถานีจาก bullet แรก
            first_line = content.split("\n")[0].strip()

            # จับชื่อสถานที่และสถานีในวงเล็บ
            name_match = re.match(r"(.+?)\s*\((.+?)\)", first_line)
            if name_match:
                name = name_match.group(1).strip(" *")
                station = name_match.group(2).strip()
            else:
                name = first_line.strip(" *")
                station = ""

            # รวมรายละเอียดทั้งหมด
            details = re.sub(r"\s*\*\s*", " ", content).strip()
            details = re.sub(r"\s+", " ", details)

            places.append({
                "name": name,
                "category": current_category,
                "station": station,
                "details": details,
                "full_text": f"[{current_category}] {name} ({station}): {details}"
            })

    return places


places = parse_tourism_data(raw_text)
print(f"พบสถานที่ทั้งหมด: {len(places)} แห่ง")
print("\n--- ตัวอย่าง 3 รายการแรก ---")
for p in places[:3]:
    print(f"  ชื่อ: {p['name']}")
    print(f"  หมวด: {p['category']}")
    print(f"  สถานี: {p['station']}")
    print(f"  full_text: {p['full_text'][:120]}...")
    print()

พบสถานที่ทั้งหมด: 54 แห่ง

--- ตัวอย่าง 3 รายการแรก ---
  ชื่อ: วัดปทุมวนารามราชวรวิหาร
  หมวด: สถานที่ทางศาสนา และวัด
  สถานี: BTS สยาม
  full_text: [สถานที่ทางศาสนา และวัด] วัดปทุมวนารามราชวรวิหาร (BTS สยาม): วัดปทุมวนารามราชวรวิหาร (BTS สยาม): ตั้งอยู่ระหว่างสยามพารา...

  ชื่อ: ศาลท้าวมหาพรหม โรงแรมเอราวัณ
  หมวด: สถานที่ทางศาสนา และวัด
  สถานี: BTS ชิดลม/สยาม
  full_text: [สถานที่ทางศาสนา และวัด] ศาลท้าวมหาพรหม โรงแรมเอราวัณ (BTS ชิดลม/สยาม): ศาลท้าวมหาพรหม โรงแรมเอราวัณ (BTS ชิดลม/สยาม): เ...

  ชื่อ: วัดธาตุทอง
  หมวด: สถานที่ทางศาสนา และวัด
  สถานี: BTS เอกมัย
  full_text: [สถานที่ทางศาสนา และวัด] วัดธาตุทอง (BTS เอกมัย): วัดธาตุทอง (BTS เอกมัย): ตั้งอยู่ใกล้ทางออก 3 ของสถานี BTS เอกมัย เดิม...



## 5. Encode — แปลงข้อมูลสถานที่เป็น Embeddings

In [6]:
documents = [p["full_text"] for p in places]

print(f"กำลัง encode {len(documents)} สถานที่...")
document_embeddings = model.encode(documents, show_progress_bar=True)

print(f"\nEncoding สำเร็จ!")
print(f"Shape ของ embeddings: {document_embeddings.shape}")
print(f"  - จำนวน documents: {document_embeddings.shape[0]}")
print(f"  - ขนาด vector (dimensions): {document_embeddings.shape[1]}")

กำลัง encode 54 สถานที่...


Batches: 100%|██████████| 2/2 [00:05<00:00,  2.56s/it]


Encoding สำเร็จ!
Shape ของ embeddings: (54, 1024)
  - จำนวน documents: 54
  - ขนาด vector (dimensions): 1024


In [7]:
# แสดงตัวอย่าง embedding ของสถานที่แรก
print(f"ตัวอย่าง embedding ของ '{places[0]['name']}':")
print(f"  Vector (5 ค่าแรก): {document_embeddings[0][:5]}")
print(f"  L2 norm: {torch.norm(torch.tensor(document_embeddings[0])).item():.4f}")

ตัวอย่าง embedding ของ 'วัดปทุมวนารามราชวรวิหาร':
  Vector (5 ค่าแรก): [ 2.3185942e-02 -6.9831694e-03 -3.8001113e-02  2.3613941e-02
  5.2445172e-05]
  L2 norm: 1.0000


## 6. บันทึก Embeddings

In [8]:
import numpy as np

# บันทึก embeddings เป็น numpy file
np.save("tourism_embeddings.npy", document_embeddings)

# บันทึก metadata เป็น JSON
with open("tourism_metadata.json", "w", encoding="utf-8") as f:
    json.dump(places, f, ensure_ascii=False, indent=2)

print("บันทึกสำเร็จ:")
print("  - tourism_embeddings.npy")
print("  - tourism_metadata.json")

บันทึกสำเร็จ:
  - tourism_embeddings.npy
  - tourism_metadata.json


## 7. Decode (Semantic Search) — ค้นหาสถานที่จาก Query

In [9]:
def semantic_search(query: str, top_k: int = 5) -> list[dict]:
    """
    ค้นหาสถานที่ที่เกี่ยวข้องกับ query โดยใช้ cosine similarity

    Args:
        query: คำถามหรือ keyword ที่ต้องการค้นหา
        top_k: จำนวนผลลัพธ์สูงสุดที่ต้องการ

    Returns:
        รายการสถานที่ที่เกี่ยวข้องที่สุด พร้อม similarity score
    """
    # BGE-M3 รองรับ query instruction prefix สำหรับ retrieval task
    instruction = "Represent this sentence for searching relevant passages: "
    query_embedding = model.encode([instruction + query])

    # คำนวณ cosine similarity
    similarities = model.similarity(query_embedding, document_embeddings)[0]

    # เรียงลำดับตาม score สูงสุด
    top_indices = torch.argsort(similarities, descending=True)[:top_k]

    results = []
    for idx in top_indices:
        idx = idx.item()
        results.append({
            **places[idx],
            "similarity_score": float(similarities[idx])
        })

    return results


def display_results(query: str, results: list[dict]):
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    for i, r in enumerate(results, 1):
        print(f"\n#{i} [{r['similarity_score']:.4f}] {r['name']}")
        print(f"   หมวด: {r['category']}")
        print(f"   สถานี: {r['station']}")
        print(f"   รายละเอียด: {r['details'][:150]}...")


print("ฟังก์ชัน semantic_search พร้อมใช้งาน")

ฟังก์ชัน semantic_search พร้อมใช้งาน


In [10]:
# Query 2: คาเฟ่
results = semantic_search("คาเฟ่บรรยากาศดี", top_k=3)
display_results("คาเฟ่บรรยากาศดี", results)


Query: 'คาเฟ่บรรยากาศดี'

#1 [0.5395] Slole Cafe & Garden
   หมวด: คาเฟ่
   สถานี: MRT ลาดพร้าว
   รายละเอียด: Slole Cafe & Garden (MRT ลาดพร้าว): สไตล์วินเทจ ตกแต่งด้วยกล้องฟิล์มวินเทจ โทนขาวคลีน มีโซนสวนด้านนอก บรรยากาศโปร่งสบาย เหมาะสำหรับสายถ่ายรูปฟิล์ม...

#2 [0.5352] Apero Cafe
   หมวด: คาเฟ่
   สถานี: MRT เพชรบุรี
   รายละเอียด: Apero Cafe (MRT เพชรบุรี): พื้นที่กะทัดรัด รองรับกลุ่มพนักงานบริษัทในช่วงพักกลางวัน ตารางสรุปคาเฟ่ยอดนิยม | รายชื่อคาเฟ่ยอดนิยม | สถานีรถไฟฟ้าที่ใกล้ท...

#3 [0.5345] HARIO CAFE
   หมวด: คาเฟ่
   สถานี: BTS ศาลาแดง
   รายละเอียด: HARIO CAFE (BTS ศาลาแดง): ตั้งอยู่ในธนิยะพลาซ่า ตกแต่งสไตล์ญี่ปุ่นผสมมินิมอล เน้นโชว์อุปกรณ์ชงกาแฟ เหมาะสำหรับคอกาแฟ Specialty...


In [11]:
# Query 3: พิพิธภัณฑ์สำหรับเด็ก
results = semantic_search("แหล่งเรียนรู้สำหรับเด็กและครอบครัว", top_k=3)
display_results("แหล่งเรียนรู้สำหรับเด็กและครอบครัว", results)


Query: 'แหล่งเรียนรู้สำหรับเด็กและครอบครัว'

#1 [0.4632] พิพิธภัณฑ์เด็กแห่งที่ 1
   หมวด: พิพิธภัณฑ์และศูนย์การเรียนรู้เชิงทัศนศึกษา
   สถานี: MRT จตุจักร/BTS หมอชิต
   รายละเอียด: พิพิธภัณฑ์เด็กแห่งที่ 1 (MRT จตุจักร/BTS หมอชิต): ตรงข้ามตลาดนัดจตุจักร เปิด อังคาร-อาทิตย์ 10:00 - 16:00 น. ฟรี...

#2 [0.4337] MOTT
   หมวด: พิพิธภัณฑ์และศูนย์การเรียนรู้เชิงทัศนศึกษา
   สถานี: MRT ลาดพร้าว
   รายละเอียด: MOTT (MRT ลาดพร้าว): พิพิธภัณฑ์เสื้อยืดวินเทจแห่งแรกของไทย ตารางสรุปพิพิธภัณฑ์และศูนย์การเรียนรู้เชิงทัศนศึกษา | สถานที่ทางวัฒนธรรมและการเรียนรู้ | สถ...

#3 [0.4273] พิพิธภัณฑ์ชาวบางกอก
   หมวด: พิพิธภัณฑ์และศูนย์การเรียนรู้เชิงทัศนศึกษา
   สถานี: BTS สุรศักดิ์
   รายละเอียด: พิพิธภัณฑ์ชาวบางกอก (BTS สุรศักดิ์): ซอยเจริญกรุง 43 บอกเล่าวิถีชีวิตชนชั้นกลางยุคสงครามโลก เปิด พุธ-อาทิตย์ 10:00 - 16:00 น. ฟรี...


In [12]:
# Query 4: ธรรมชาติและพื้นที่สีเขียว
results = semantic_search("สวนสาธารณะและพื้นที่สีเขียวพักผ่อน", top_k=3)
display_results("สวนสาธารณะและพื้นที่สีเขียวพักผ่อน", results)


Query: 'สวนสาธารณะและพื้นที่สีเขียวพักผ่อน'

#1 [0.5053] สวนเบญจกิติ
   หมวด: พื้นที่สีเขียว และจุดชมวิวเมือง
   สถานี: MRT สุขุมวิท/BTS อโศก
   รายละเอียด: สวนเบญจกิติ (MRT สุขุมวิท/BTS อโศก): สวนป่าเชิงนิเวศ (Forest Park) ขนาดกว่า 450 ไร่ เปิด 05:00 - 21:00 น....

#2 [0.4951] อุทยาน 100 ปี จุฬาฯ
   หมวด: พื้นที่สีเขียว และจุดชมวิวเมือง
   สถานี: BTS สยาม/สนามกีฬา
   รายละเอียด: อุทยาน 100 ปี จุฬาฯ (BTS สยาม/สนามกีฬา): สวนหน่วงน้ำที่ออกแบบเพื่อรองรับปัญหาน้ำท่วม เปิด 05:00 - 22:00 น....

#3 [0.4839] Papaya Design Furniture and Studio
   หมวด: พื้นที่สีเขียว และจุดชมวิวเมือง
   สถานี: MRT ลาดพร้าว
   รายละเอียด: Papaya Design Furniture and Studio (MRT ลาดพร้าว): โกดังเรโทร 4 ชั้น เข้าชมฟรี...


In [13]:
# Query 5: ช้อปปิ้ง
results = semantic_search("ห้างสรรพสินค้าสำหรับนักท่องเที่ยวต่างชาติ", top_k=3)
display_results("ห้างสรรพสินค้าสำหรับนักท่องเที่ยวต่างชาติ", results)


Query: 'ห้างสรรพสินค้าสำหรับนักท่องเที่ยวต่างชาติ'

#1 [0.4774] Marche Thonglor
   หมวด: ศูนย์การค้า ตลาด และห้างสรรพสินค้า
   สถานี: BTS ทองหล่อ
   รายละเอียด: Marche Thonglor (BTS ทองหล่อ): ซอย 10 มิกซ์ยูสพร้อมพื้นที่สีเขียว ห่างสถานี 600 เมตร เปิดถึงเที่ยงคืน...

#2 [0.4741] Don Don Donki Thonglor
   หมวด: ศูนย์การค้า ตลาด และห้างสรรพสินค้า
   สถานี: BTS ทองหล่อ
   รายละเอียด: Don Don Donki Thonglor (BTS ทองหล่อ): ซูเปอร์มาร์เก็ตญี่ปุ่นเปิด 24 ชม....

#3 [0.4736] The Commons Thonglor
   หมวด: ศูนย์การค้า ตลาด และห้างสรรพสินค้า
   สถานี: BTS ทองหล่อ
   รายละเอียด: The Commons Thonglor (BTS ทองหล่อ): ซอยทองหล่อ 17 สถาปัตยกรรมลอฟท์กึ่งเปิดประทุน เปิด 08:00 - 01:00 น....


In [14]:
# Query 6: ภาษาอังกฤษ
results = semantic_search("free admission historical temple Bangkok", top_k=3)
display_results("free admission historical temple Bangkok", results)


Query: 'free admission historical temple Bangkok'

#1 [0.4996] วัดปทุมวนารามราชวรวิหาร
   หมวด: สถานที่ทางศาสนา และวัด
   สถานี: BTS สยาม
   รายละเอียด: วัดปทุมวนารามราชวรวิหาร (BTS สยาม): ตั้งอยู่ระหว่างสยามพารากอนและเซ็นทรัลเวิลด์ ก่อตั้งปี พ.ศ. 2400 ในรัชกาลที่ 4 ห่างจากสถานี BTS สยาม ทางออก 5 เพียง...

#2 [0.4842] พิพิธภัณฑ์ชาวบางกอก
   หมวด: พิพิธภัณฑ์และศูนย์การเรียนรู้เชิงทัศนศึกษา
   สถานี: BTS สุรศักดิ์
   รายละเอียด: พิพิธภัณฑ์ชาวบางกอก (BTS สุรศักดิ์): ซอยเจริญกรุง 43 บอกเล่าวิถีชีวิตชนชั้นกลางยุคสงครามโลก เปิด พุธ-อาทิตย์ 10:00 - 16:00 น. ฟรี...

#3 [0.4719] Madame Tussauds Bangkok
   หมวด: พิพิธภัณฑ์และศูนย์การเรียนรู้เชิงทัศนศึกษา
   สถานี: BTS สยาม
   รายละเอียด: Madame Tussauds Bangkok (BTS สยาม): ห่างสถานี 370 เมตร (เดิน 4 นาที) ชั้น 4 สยามดิสคัฟเวอรี่ เปิด 10:00 - 20:00 น....


## 8. เปรียบเทียบ Similarity Matrix ระหว่างสถานที่

In [15]:
# เลือกสถานที่ตัวแทนจากแต่ละหมวด
sample_names = [
    "วัดปทุมวนารามราชวรวิหาร",
    "SEA LIFE Bangkok Ocean World",
    "The Commons Thonglor",
    "สวนเบญจกิติ",
    "Patom Organic Living",
]

sample_indices = []
for name in sample_names:
    for i, p in enumerate(places):
        if name in p["name"]:
            sample_indices.append(i)
            break

sample_embeddings = document_embeddings[sample_indices]
sim_matrix = model.similarity(
    torch.tensor(sample_embeddings),
    torch.tensor(sample_embeddings)
)

print("Cosine Similarity Matrix:")
header = [f"{places[i]['name'][:12]}" for i in sample_indices]
print(f"{'':15}" + "".join(f"{h:>15}" for h in header))
for i, row_idx in enumerate(sample_indices):
    row_name = places[row_idx]['name'][:14]
    scores = "".join(f"{sim_matrix[i][j].item():>15.4f}" for j in range(len(sample_indices)))
    print(f"{row_name:<15}{scores}")

Cosine Similarity Matrix:
                  วัดปทุมวนารา   SEA LIFE Ban   The Commons     สวนเบญจกิติ   Patom Organi
วัดปทุมวนารามร          1.0000         0.7070         0.5456         0.5624         0.5624
SEA LIFE Bangk          0.7070         1.0000         0.6117         0.6274         0.6355
The Commons Th          0.5456         0.6117         1.0000         0.5822         0.6959
สวนเบญจกิติ             0.5624         0.6274         0.5822         1.0000         0.6194
Patom Organic           0.5624         0.6355         0.6959         0.6194         1.0000


## 9. Interactive Search

In [17]:
my_query = "วัดใกล้ BTS สยาม"
top_k = 5

results = semantic_search(my_query, top_k=top_k)
display_results(my_query, results)


Query: 'วัดใกล้ BTS สยาม'

#1 [0.6220] วัดปทุมวนารามราชวรวิหาร
   หมวด: สถานที่ทางศาสนา และวัด
   สถานี: BTS สยาม
   รายละเอียด: วัดปทุมวนารามราชวรวิหาร (BTS สยาม): ตั้งอยู่ระหว่างสยามพารากอนและเซ็นทรัลเวิลด์ ก่อตั้งปี พ.ศ. 2400 ในรัชกาลที่ 4 ห่างจากสถานี BTS สยาม ทางออก 5 เพียง...

#2 [0.5563] Madame Tussauds Bangkok
   หมวด: พิพิธภัณฑ์และศูนย์การเรียนรู้เชิงทัศนศึกษา
   สถานี: BTS สยาม
   รายละเอียด: Madame Tussauds Bangkok (BTS สยาม): ห่างสถานี 370 เมตร (เดิน 4 นาที) ชั้น 4 สยามดิสคัฟเวอรี่ เปิด 10:00 - 20:00 น....

#3 [0.5330] วัดลาดพร้าว
   หมวด: สถานที่ทางศาสนา และวัด
   สถานี: MRT ลาดพร้าว
   รายละเอียด: วัดลาดพร้าว (MRT ลาดพร้าว): โดดเด่นด้วยคอลเลกชันพระพุทธรูปปางต่างๆ จำนวนมาก...

#4 [0.5272] ศาลท้าวมหาพรหม โรงแรมเอราวัณ
   หมวด: สถานที่ทางศาสนา และวัด
   สถานี: BTS ชิดลม/สยาม
   รายละเอียด: ศาลท้าวมหาพรหม โรงแรมเอราวัณ (BTS ชิดลม/สยาม): เป็นจุดตัดระหว่างความเชื่อฮินดูพราหมณ์และการท่องเที่ยวใจกลางเมือง เปิดทุกวัน 06:00 - 22:00 น....

#5 [0.5237] วัดธาตุทอง
   หมวด: สถานที่ทาง